# Nadal vs Federer — Probability of Winning from Deuce

Nadal and Federer are playing a tennis game.

Given:

- Probability Nadal wins a point: `p = 0.7`
- Probability Federer wins a point: `q = 0.3`
- Current score: **Deuce**

Question:

> What is the probability that Nadal wins the game from Deuce?

This is **not a Bayes' Rule problem**. It is a recursive probability / Markov process problem.

We solve it using three approaches:

1. Recursive state equations
2. Deuce-cycle reasoning
3. Simulation


## Setup

In [1]:
p = 0.7
q = 1 - p

p, q

(0.7, 0.30000000000000004)

## Approach 1 — Recursive State Equations

Define three probabilities:

- `D` = probability Nadal wins the game starting from Deuce
- `AN` = probability Nadal wins starting from Advantage Nadal
- `AF` = probability Nadal wins starting from Advantage Federer

From Deuce:

$$D = p \cdot AN + q \cdot AF$$

From Advantage Nadal:

$$AN = p + qD$$

From Advantage Federer:

$$AF = pD$$

Substitute into the Deuce equation:

$$D = p(p + qD) + q(pD)$$

Solving:

$$D = p^2 + 2pqD$$

$$D(1 - 2pq) = p^2$$

$$D = \frac{p^2}{1 - 2pq}$$


In [2]:
d_recursive = p**2 / (1 - 2 * p * q)
d_recursive

0.8448275862068965

So the probability that Nadal wins the game from Deuce is approximately:

$$0.8448$$

or **84.48%**.


## Approach 2 — Deuce-Cycle Reasoning

From Deuce, look at the next two points.

### Nadal wins both points

$$p^2$$

Nadal wins the game immediately.

### Federer wins both points

$$q^2$$

Federer wins the game immediately.

### They split the two points

This can happen in two ways:

- Nadal then Federer
- Federer then Nadal

Probability:

$$2pq$$

In this case, the game returns to Deuce.

So if `D` is the probability Nadal eventually wins from Deuce:

$$D = p^2 + 2pqD$$

Therefore:

$$D = \frac{p^2}{1 - 2pq}$$


In [3]:
d_cycle = p**2 / (1 - 2 * p * q)
d_cycle

0.8448275862068965

This gives the same answer as the recursive state-equation method.


## Approach 3 — Point-by-Point Simulation

Now simulate many games starting from Deuce.

Rules:

- At Deuce, a player must win two consecutive points to win the game.
- If players split the next two points, the score returns to Deuce.


In [4]:
import numpy as np

def simulate_game_from_deuce(
    p_nadal: float,
    rng: np.random.Generator,
) -> bool:
    """
    Return True if Nadal wins the game from Deuce.
    Return False if Federer wins.
    """
    state = "Deuce"

    while True:
        nadal_wins_point = rng.random() < p_nadal

        if state == "Deuce":
            if nadal_wins_point:
                state = "Advantage Nadal"
            else:
                state = "Advantage Federer"

        elif state == "Advantage Nadal":
            if nadal_wins_point:
                return True
            else:
                state = "Deuce"

        elif state == "Advantage Federer":
            if nadal_wins_point:
                state = "Deuce"
            else:
                return False

In [5]:
n = 100_000
rng = np.random.default_rng(42)

wins = np.array([
    simulate_game_from_deuce(p, rng)
    for _ in range(n)
])

simulated_probability = wins.mean()
simulated_probability

np.float64(0.84483)

The simulated probability should be close to the exact answer:

$$0.8448$$


## Faster Simulation by Deuce Cycles

Instead of simulating point-by-point, we can simulate each deuce cycle.

From Deuce, the next two points can result in:

- Nadal wins game: probability `p²`
- Federer wins game: probability `q²`
- Return to Deuce: probability `2pq`

This is faster than point-by-point simulation.


In [6]:
def simulate_deuce_cycles(
    p_nadal: float,
    n_games: int,
    seed: int | None = None,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    q_federer = 1 - p_nadal

    outcomes = np.empty(n_games, dtype=bool)
    unresolved = np.ones(n_games, dtype=bool)

    p_nadal_wins_cycle = p_nadal ** 2
    p_federer_wins_cycle = q_federer ** 2

    while unresolved.any():
        idx = np.where(unresolved)[0]
        u = rng.random(len(idx))

        nadal_win = u < p_nadal_wins_cycle

        federer_win = (
            u >= p_nadal_wins_cycle
        ) & (
            u < p_nadal_wins_cycle + p_federer_wins_cycle
        )

        outcomes[idx[nadal_win]] = True
        outcomes[idx[federer_win]] = False

        resolved = nadal_win | federer_win
        unresolved[idx[resolved]] = False

    return outcomes

In [7]:
wins_fast = simulate_deuce_cycles(
    p_nadal=p,
    n_games=1_000_000,
    seed=42,
)

cycle_simulated_probability = wins_fast.mean()
cycle_simulated_probability

np.float64(0.844689)

## Compare All Three Approaches

In [8]:
import pandas as pd

comparison = pd.DataFrame(
    {
        "Approach": [
            "Recursive equations",
            "Deuce-cycle formula",
            "Point-by-point simulation",
            "Cycle simulation",
        ],
        "Probability Nadal wins": [
            d_recursive,
            d_cycle,
            simulated_probability,
            cycle_simulated_probability,
        ],
    }
)

comparison

,Approach,Probability Nadal wins
0,Recursive equations,0.844828
1,Deuce-cycle formula,0.844828
2,Point-by-point simulation,0.844830
3,Cycle simulation,0.844689


## Final Answer

The probability that Nadal wins the game from Deuce is:

$$
\frac{p^2}{1 - 2p(1-p)}
$$

With:

$$p = 0.7$$

we get:

$$
P(\text{Nadal wins from Deuce})
=
\frac{0.7^2}{1 - 2(0.7)(0.3)}
=
0.8448
$$

So Nadal has about an **84.48%** chance of winning the game from Deuce.


## Key Takeaways

1. This is not a Bayes' Rule problem.
2. It is a recursive probability / Markov process problem.
3. The score can return to Deuce repeatedly.
4. Recursive equations and simulation are both natural approaches.
5. The deuce-cycle method gives a compact closed-form solution.
